# Map de novo mutations from parent-offspring trios

---
# 1. Format DNM data file

## 1a: Roulette dataset
Data from Seplyarskiy et al., 2023, publicaly available at http://genetics.bwh.harvard.edu/downloads/Vova/Neural_net_tracks/all_mut

In [ ]:
# Define input and output file paths
input_file_path = 'roulette_dnm_GRCh38.txt'
output_file_path = 'roulette_dnm_GRCh38_defined.txt'

# Open the input file and read lines
with open(input_file_path, 'r') as file:
    lines = file.readlines()

# Process each line
processed_lines = []
for line in lines:
    # Split the line by whitespace
    columns = line.split()
    # Extract the third column and process it
    original_string = columns[2]
    new_string = f"{original_string[1]}>{original_string[-1]}"
    # Replace the third column with the new string
    columns[2] = new_string
    # Join the columns back into a single line
    processed_line = ' '.join(columns)
    # Append the processed line to the list
    processed_lines.append(processed_line)

# Write the processed lines to the output file
with open(output_file_path, 'w') as file:
    for processed_line in processed_lines:
        file.write(processed_line + '\n')

### Additionally need to convert mutations with respect to coding strand

In [ ]:
import subprocess
import os

def get_reference_nucleotide(chrom, pos, fasta_file):
    """Use samtools faidx to get the reference nucleotide at a position"""
    # Note: samtools uses 1-based positioning
    region = f"{chrom}:{pos}-{pos}"
    
    try:
        result = subprocess.run(
            ['samtools', 'faidx', fasta_file, region],
            check=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True
        )
        
        # Parse the output (header line followed by sequence)
        lines = result.stdout.split('\n')
        if len(lines) >= 2:
            return lines[1].strip().upper()
        return 'N'
    except subprocess.CalledProcessError:
        return 'N'

def main():
    # Hardcoded file names
    input_file = 'roulette_dnm_GRCh38_defined.txt'  # Change this if your input has a different name
    output_file = 'roulette_dnm_GRCh38_defined_refnuc.txt'
    fasta_file = '/media/alexpalazzo1/ohta/Tina/SNPs_and_Indels/PrimateSNPs/Homo_sapiens.GRCh38.dna.toplevel.fa'  # Or .fa.gz if compressed
    
    # Check if files exist
    if not os.path.exists(input_file):
        print(f"Error: Input file '{input_file}' not found.")
        return
    if not os.path.exists(fasta_file):
        print(f"Error: Reference genome file '{fasta_file}' not found.")
        return
    
    # Check if samtools is available
    try:
        subprocess.run(['samtools', '--version'], check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    except (subprocess.CalledProcessError, FileNotFoundError):
        print("Error: samtools is not installed or not in PATH")
        print("Please install samtools (conda install -c bioconda samtools)")
        return
    
    # Create index if it doesn't exist
    if not os.path.exists(fasta_file + '.fai'):
        print("Creating FASTA index...")
        subprocess.run(['samtools', 'faidx', fasta_file], check=True)
    
    # Process input file
    with open(input_file, 'r') as infile, open(output_file, 'w') as outfile:
        for line in infile:
            fields = line.strip().split()
            if len(fields) < 3:
                continue  # skip malformed lines
            
            chrom = fields[0]
            pos = int(fields[1])
            
            # Get the reference nucleotide
            ref_nuc = get_reference_nucleotide(chrom, pos, fasta_file)
            
            # Write the original line with the reference nucleotide added
            outfile.write('\t'.join(fields + [ref_nuc]) + '\n')
    
    print(f"Processing complete. Output written to {output_file}")

if __name__ == '__main__':
    main()


def reverse_complement_mutation(mutation):
    """Convert a mutation to its reverse complement"""
    comp = {'A': 'T', 'T': 'A', 'G': 'C', 'C': 'G'}
    ref, alt = mutation.split('>')
    return f"{comp[ref]}>{comp[alt]}"

def process_file(input_filename):
    """Process the input file according to the specified rules"""
    output_lines = []
    error_lines = []
    
    # Define reverse complement pairs
    reverse_pairs = {'A': 'T', 'T': 'A', 'G': 'C', 'C': 'G'}
    
    with open(input_filename, 'r') as f:
        for line in f:
            fields = line.strip().split()
            if len(fields) < 6:
                error_lines.append(line)
                continue
                
            chrom, pos, mutation, sample_id, qual, second_last, last = fields[:7]
            ref_allele = mutation[0]
            
            # Check if reference allele matches last column
            if ref_allele == last:
                output_lines.append(line)
                continue
                
            # Check if it's a reverse complement mismatch and second last column is 'D'
            if (second_last == 'D' and 
                last in reverse_pairs and 
                reverse_pairs[ref_allele] == last):
                # Perform reverse complement of the mutation
                new_mutation = reverse_complement_mutation(mutation)
                new_line = '\t'.join([chrom, pos, new_mutation] + fields[3:]) + '\n'
                output_lines.append(new_line)
            else:
                error_lines.append(line)
    
    # Write output file (same as input but with some mutations changed)
    with open('roulette_dnm_GRCh38_defined_refnuc_corrected.txt', 'w') as f:
        f.writelines(output_lines)
    
    # Write error file
    with open('roulette_dnm_GRCh38_defined_refnuc_error.txt', 'w') as f:
        f.writelines(error_lines)

# Hardcoded input filename
input_filename = 'roulette_dnm_GRCh38_defined_refnuc.txt'
process_file(input_filename)

## 1b: Decode dataset
Data from Palsson et al., 2025, publically available at https://zenodo.org/records/14025565

In [ ]:
# Read and process the file
with open('dnms.tsv', 'r') as infile:
    lines = infile.readlines()

# Process lines, skipping first 10 and joining columns 3-4 with '>'
output_lines = []
for line in lines[10:]:  # Skip first 10 lines
    if line.strip():  # Skip empty lines
        columns = line.strip().split('\t')
        if len(columns) >= 4:
            ref_alt = f"{columns[2]}>{columns[3]}"
            new_line = '\t'.join([columns[0], columns[1], ref_alt] + columns[4:])
            output_lines.append(new_line)

# Write to output file
with open('decode_dnm.txt', 'w') as outfile:
    outfile.write('\n'.join(output_lines))

print("File reformatted successfully!")

---
# 2. Map DNMs
Map DNMs to relative positions 1kb around the TSS

In [ ]:
import concurrent.futures
import logging

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

def read_file2(file_path):
    data = {}
    try:
        with open(file_path, 'r') as file:
            for line in file:
                parts = line.strip().split()
                chr_num = parts[0].replace('chr', '')  # Remove "chr" prefix
                start = int(parts[1])
                end = int(parts[2])
                direction = parts[3]  # New column for direction

                # Store data correctly based on direction
                if chr_num not in data:
                    data[chr_num] = []
                data[chr_num].append((start, end, direction))
    except Exception as e:
        logging.error(f"Error reading file2: {e}")
        raise
    return data

def process_chunk(chunk, file2_data):
    results = []
    for line in chunk:
        try:
            parts = line.strip().split()
            chr_num = parts[0].replace('chr', '')  # Remove "chr" prefix
            coord = int(parts[1])
            extra_data = parts[2:]
            
            if chr_num in file2_data:
                for start, end, direction in file2_data[chr_num]:
                    if direction == '+':
                        if start <= coord <= end:
                            range_position = coord - start
                            results.append(f"{range_position} {' '.join(extra_data)} {chr_num} {coord}\n")
                            break
                    elif direction == '-':
                        if end <= coord <= start:
                            range_position = start - coord
                            results.append(f"{range_position} {' '.join(extra_data)} {chr_num} {coord}\n")
                            break
                    else:
                        raise ValueError(f"Invalid direction found: {direction}")
        except Exception as e:
            logging.error(f"Error processing line: {line.strip()}. Error: {e}")
    return results

def process_files(file1_path, file2_path, output_file_path, chunk_size=100000):
    file2_data = read_file2(file2_path)

    try:
        with open(file1_path, 'r') as file1, open(output_file_path, 'w') as output_file:
            next(file1)  # Skip the first line (title line)
            
            with concurrent.futures.ProcessPoolExecutor() as executor:
                chunk = []
                futures = []
                for line in file1:
                    chunk.append(line)
                    if len(chunk) == chunk_size:
                        futures.append(executor.submit(process_chunk, chunk, file2_data))
                        chunk = []
                if chunk:
                    futures.append(executor.submit(process_chunk, chunk, file2_data))

                for future in concurrent.futures.as_completed(futures):
                    results = future.result()
                    output_file.writelines(results)
            
        logging.info("File processing completed successfully.")
    except Exception as e:
        logging.error(f"Error processing files: {e}")

# Paths to input files and output file
file1_path = 'Decode/decode_dnm.txt'
file2_path = 'human_intergenic_random_1kb.txt'
output_file_path = 'Decode/decode_dnm_GRCh38_random_intergenic_mapped.txt'

# Process the files
process_files(file1_path, file2_path, output_file_path)

### For Decode dataset
Since Decode dataset contains indels and DNMs, will filter out the indels

In [ ]:
def filter_indel_lines(input_file, output_file):
    """
    Reads input file and removes lines that have 'Indel' in the 5th column.
    
    Args:
        input_file (str): Path to the input file
        output_file (str): Path to the output file
    """
    try:
        with open(input_file, 'r') as infile, open(output_file, 'w') as outfile:
            for line in infile:
                # Split the line into columns
                columns = line.strip().split()
                
                # Check if there are enough columns and if 5th column is not "Indel"
                if len(columns) >= 5 and columns[4] != "Indel":
                    outfile.write(line)
        
        print(f"Successfully filtered file. Output written to {output_file}")
        
    except FileNotFoundError:
        print(f"Error: Input file '{input_file}' not found.")
    except Exception as e:
        print(f"An error occurred: {e}")

# Example usage
input_filename = "decode_dnm_GRCh38_random_intergenic_mapped.txt"  # Replace with your input file name
output_filename = "decode_dnm_GRCh38_random_intergenic_mapped_noindel.txt"  # Replace with your desired output file name

filter_indel_lines(input_filename, output_filename)

---
# 3. Count DNMs
Count total number of DNMs for each position 1kb around the TSS for each mutation type

In [ ]:
import pandas as pd

# Define the path to the file
file_path = 'Decode/decode_dnm_GRCh38_testis_lncRNA_mapped_noindel.txt'

# Read the file
with open(file_path, 'r') as file:
    lines = file.readlines()

# Initialize a dictionary to keep track of counts
mutation_types = ["A>G", "T>C", "C>G", "T>G", "C>A", "A>T", "G>C", "G>T", "C>T", "T>A", "A>C", "G>A"]
counts = {i: {mutation: 0 for mutation in mutation_types} for i in range(1, 1001)}

# Process each line
for line in lines:
    parts = line.split()
    if parts[0] == "NA":
        continue  # Skip this line
    row_index = int(parts[0])
    if 1 <= row_index <= 999:  # Ensure the row index is within the valid range (1-999)
        row_index -= 0  # Convert to 1-based index and add 1 for header
        mutation_type = parts[1]
        if mutation_type in mutation_types:
            counts[row_index][mutation_type] += 1

# Convert the counts dictionary to a DataFrame
df = pd.DataFrame.from_dict(counts, orient='index', columns=mutation_types)


# Save the DataFrame to a CSV file
output_file_path = 'Decode/decode_dnm_GRCh38_testis_lncRNA_mapped_noindel_count.csv'
df.to_csv(output_file_path, index_label="Position")